# 03 · Feature Engineering
**Ninja Van Capstone 2026 — Last-Mile Delivery Failure Prediction**

This notebook:
1. Loads raw LaDe data
2. Applies all feature engineering steps (temporal, courier, address, geo, parcel, recipient)
3. Fits and saves the sklearn `ColumnTransformer` preprocessor
4. Produces train/test splits as parquet files
5. Optionally generates a SMOTE-balanced training set

**Outputs:**
- `/content/data/processed/features_train.parquet`
- `/content/data/processed/features_test.parquet`
- `/content/data/processed/features_train_smote.parquet` *(if use_smote=true)*
- `/content/outputs/preprocessor.joblib`
- `/content/outputs/feature_names.json`
- `/content/outputs/class_weights.json`

> **Run order:** Run after `02_eda.ipynb`.


## 1 · Setup

In [ ]:
import sys, json, joblib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO_DIR = Path("/content/nv-lastmile-training")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_DIR    = Path("/content/data/raw")
PROC_DIR   = Path("/content/data/processed")
OUT_DIR    = Path("/content/outputs")
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load config
with open(REPO_DIR / "config" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

RANDOM_STATE = cfg["data"]["random_state"]
TEST_SIZE    = cfg["data"]["test_size"]
USE_SMOTE    = cfg["features"]["use_smote"]

print(f"Config loaded — test_size={TEST_SIZE}, random_state={RANDOM_STATE}, use_smote={USE_SMOTE}")

## 2 · Load Raw LaDe Data

In [ ]:
from data_loader import load_raw

df_raw = load_raw("lade", str(RAW_DIR))
print(f"Raw LaDe: {len(df_raw):,} rows × {len(df_raw.columns)} columns")
print(df_raw.dtypes.to_string())

## 3 · Build Feature Matrix

This calls the full feature engineering pipeline from `src/features.py`. Steps run in order:
1. Derive `delivery_failed` target
2. Temporal features
3. Address features
4. Geo + parcel bucket features
5. Recipient expanding-window features
6. Courier expanding-window features
7. Select final 20-column feature matrix

In [ ]:
from features import (
    build_full_feature_matrix,
    ALL_FEATURE_COLS,
    NUMERIC_FEATURES,
    BINARY_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COL,
)

X, y = build_full_feature_matrix(df_raw)

print(f"\nFeature matrix shape : {X.shape}")
print(f"Target distribution  : {y.value_counts().to_dict()}")
print(f"Failure rate         : {y.mean():.4f}")
print(f"\nFeature columns ({len(ALL_FEATURE_COLS)}):")
for i, col in enumerate(ALL_FEATURE_COLS):
    print(f"  {i+1:2d}. {col}")

## 4 · Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train : {len(X_train):,} rows  (failure rate: {y_train.mean():.4f})")
print(f"Test  : {len(X_test):,} rows  (failure rate: {y_test.mean():.4f})")

## 5 · Fit Preprocessor

In [ ]:
from features import get_preprocessor

preprocessor = get_preprocessor(
    numeric_cols=NUMERIC_FEATURES + BINARY_FEATURES,
    categorical_cols=CATEGORICAL_FEATURES,
)

# Fit on training data ONLY
preprocessor.fit(X_train)
print("✅ Preprocessor fitted on training set")

# Quick sanity-check: transform a single row
X_train_t = preprocessor.transform(X_train)
X_test_t  = preprocessor.transform(X_test)
print(f"   Transformed train shape : {X_train_t.shape}")
print(f"   Transformed test  shape : {X_test_t.shape}")

## 6 · Compute Class Weights

In [ ]:
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
ratio = n_neg / max(n_pos, 1)

class_weights = {0: 1.0, 1: round(ratio, 4)}

print(f"Negative (success) : {n_neg:,}")
print(f"Positive (failure) : {n_pos:,}")
print(f"Imbalance ratio    : {ratio:.2f}:1")
print(f"Class weights      : {class_weights}")

# Save for use in training notebook
with open(OUT_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=2)
print(f"\n✅ Class weights saved → {OUT_DIR / 'class_weights.json'}")

## 7 · SMOTE Balanced Training Set (Optional)

In [ ]:
if USE_SMOTE:
    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=RANDOM_STATE)
        X_train_sm, y_train_sm = sm.fit_resample(X_train_t, y_train)
        print(f"✅ SMOTE applied:")
        print(f"   Before: {X_train_t.shape[0]:,} rows  (failure rate: {y_train.mean():.4f})")
        print(f"   After : {X_train_sm.shape[0]:,} rows  (failure rate: {y_train_sm.mean():.4f})")
        # Store as DataFrame with feature names
        feature_names_out = preprocessor.get_feature_names_out().tolist()
        X_sm_df = pd.DataFrame(X_train_sm, columns=feature_names_out)
        X_sm_df[TARGET_COL] = y_train_sm.values
        X_sm_df.to_parquet(PROC_DIR / "features_train_smote.parquet", index=False)
        print(f"   Saved → features_train_smote.parquet")
    except ImportError:
        print("⚠️  imbalanced-learn not installed. Run: pip install imbalanced-learn")
else:
    print("ℹ️  use_smote=false in config — skipping SMOTE")

## 8 · Save Processed Splits as Parquet

In [ ]:
# Save raw (un-transformed) splits with target column
# The training notebook will apply the preprocessor.transform() at runtime
X_train_save = X_train.copy()
X_train_save[TARGET_COL] = y_train.values
X_train_save.to_parquet(PROC_DIR / "features_train.parquet", index=False)

X_test_save = X_test.copy()
X_test_save[TARGET_COL] = y_test.values
X_test_save.to_parquet(PROC_DIR / "features_test.parquet", index=False)

print(f"✅ Saved:")
print(f"   {PROC_DIR / 'features_train.parquet'}  ({len(X_train_save):,} rows)")
print(f"   {PROC_DIR / 'features_test.parquet'}   ({len(X_test_save):,} rows)")

## 9 · Save Preprocessor & Feature Names

In [ ]:
# Save preprocessor
preprocessor_path = OUT_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path, compress=3)
print(f"✅ Preprocessor saved → {preprocessor_path}")

# Save feature names list (the contract between repos)
feature_names_path = OUT_DIR / "feature_names.json"
with open(feature_names_path, "w") as f:
    json.dump(ALL_FEATURE_COLS, f, indent=2)
print(f"✅ Feature names saved → {feature_names_path}")
print(f"   Features: {ALL_FEATURE_COLS}")

## 10 · Feature Distribution Preview

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 14))
axes = axes.flatten()

for i, col in enumerate(ALL_FEATURE_COLS):
    ax = axes[i]
    col_data = X_train[col].dropna()

    if col in CATEGORICAL_FEATURES:
        vc = col_data.astype(str).value_counts().head(8)
        ax.bar(vc.index, vc.values, color="#457B9D", edgecolor="white")
        ax.tick_params(axis="x", rotation=30, labelsize=7)
    else:
        ax.hist(col_data, bins=30, color="#457B9D", edgecolor="white", alpha=0.85)

    ax.set_title(col, fontsize=8, fontweight="bold")
    ax.set_ylabel("Count", fontsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide unused subplots
for j in range(len(ALL_FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions (Training Set)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
OUT_DIR_EDA = OUT_DIR / ".." / "outputs" / "eda"
import os; os.makedirs(str(OUT_DIR / "eda"), exist_ok=True)
fig.savefig(OUT_DIR / "eda" / "08_feature_distributions.png", bbox_inches="tight", dpi=130)
plt.show()
print("✅ Feature distributions plot saved")

In [ ]:
print("=" * 70)
print("  NOTEBOOK 03 COMPLETE")
print("=" * 70)
print(f"  Features train  : {PROC_DIR / 'features_train.parquet'}")
print(f"  Features test   : {PROC_DIR / 'features_test.parquet'}")
print(f"  Preprocessor    : {OUT_DIR / 'preprocessor.joblib'}")
print(f"  Feature names   : {OUT_DIR / 'feature_names.json'}")
print(f"  Class weights   : {OUT_DIR / 'class_weights.json'}")
print("\n✅ Proceed to 04_model_training.ipynb")